# Домашняя работа: квантизация и дистилляция LLM

В этой работе вы самостоятельно проведёте эксперименты с квантизацией и дистилляцией LLM и проверите, сколько качества при этом теряет модель. Задача, на которую обучаемся, — разбор запросов к голосовому ассистенту из датасета MASSIVE (ru-RU): по запросу модель возвращает JSON с intent и slots. Подробнее: https://huggingface.co/datasets/AmazonScience/massive

### Окружение

Зависимости зафиксированы в requirements.txt. Ноутбук запускается из папки hw7. Результаты сохраняются после каждого прогона, поэтому при повторном открытии можно сразу посмотреть таблицы.

In [1]:
# %pip install -r requirements.txt

In [2]:
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer

from experiment import (
    ARTIFACTS, BASE_MODEL, BASE_REVISION, TEACHER_MODEL, TEACHER_REVISION,
    METRICS, QUANT, STUDENTS, TeacherAnswerDataset,
    done, encode_prompt, read_rows, select_student,
)
from massive_data import (
    DATASET_REVISION, SEED, compute_metrics, load_massive,
    make_system_prompt, prepare_massive,
)

<venv>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def run(name):
    if not done(name):
        subprocess.run([sys.executable, "experiment.py", name], check=True)


def result(name):
    return json.loads((ARTIFACTS / "eval" / name / "metrics.json").read_text())


def scores(names):
    return pd.DataFrame([result(name) for name in names]).set_index("run")[METRICS].round(4)


print(ARTIFACTS.name)

artifacts_compression_hw


## Загрузка MASSIVE

Используем датасет MASSIVE. Все калибровочные примеры и данные для дистилляции берём только из train. Оцениваем на валидации (dev).

In [4]:
raw = load_massive()
print(DATASET_REVISION)
print({split: len(rows) for split, rows in raw.items()})

ed58ac423a2f4121720918bf5301577edce4ffd3
{'train': 11514, 'validation': 2033, 'test': 2974}


In [5]:
example = raw["train"][0]
print(example["utt"])
print(example["annot_utt"])

разбуди меня в девять утра в пятницу
разбуди меня в [time : девять утра] в [date : пятницу]


### Сконвертируем данные для калибровки

В MASSIVE текст запроса лежит в поле `utt`, intent записан числом, а slots отмечены прямо в тексте поля `annot_utt`, например `[time : семь утра]`. Модель должна по запросу вернуть JSON с названием intent и списком slots в виде пар «тип — значение». Поэтому преобразуем запись в `text` — исходный запрос — и `gold` — эталонный JSON-ответ. Такой формат удобнее для калибровки и подсчёта метрик.

In [6]:
train_data, dev_data, intents, slot_types = prepare_massive(raw)

In [7]:
print(f"{len(intents)} intents, {len(slot_types)} slot types")
print(f"{len(train_data)} train, {len(dev_data)} dev, seed {SEED}")

60 intents, 54 slot types
11514 train, 256 dev, seed 42


In [8]:
train_data[0]

{'id': '1',
 'text': 'разбуди меня в девять утра в пятницу',
 'gold': '{"intent":"alarm_set","slots":[["time","девять утра"],["date","пятницу"]]}'}

### Генерация ответов
Всем моделям даём одинаковую инструкцию и одинаковые описания допустимых классов. Генерация детерминирована: greedy decoding, одинаковый лимит токенов.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
system_prompt = make_system_prompt(intents, slot_types)
print("бейзлайн и ученик", BASE_MODEL)

бейзлайн и ученик Qwen/Qwen3-1.7B


In [10]:
prompt_ids = encode_prompt(tokenizer, system_prompt, train_data[0]["text"])
print("токенов в первом промпте", len(prompt_ids))

токенов в первом промпте 504


In [11]:
run("baseline")

## Метрика качества

$$
Q = \tfrac{1}{2} \operatorname{IntentAccuracy} + \tfrac{1}{2} \operatorname{SlotMicroF1}.
$$

Slots сравниваются как **мультимножество** пар `(тип, значение)`: порядок не влияет, лишние дубликаты дают false positives. Неверный JSON или нарушение схемы считается ошибкой всего ответа.

Дополнительно считаем `frame_exact_match` и `valid_json_rate`. Если во всей выборке нет ни одного gold/predicted slot, slot F1 равен 1 только при валидности всех ответов, иначе 0.

In [12]:
predictions = read_rows(ARTIFACTS / "eval" / "baseline" / "predictions.jsonl")
baseline_metrics = compute_metrics(
    [row["prediction"] for row in predictions],
    dev_data["gold"], intents, slot_types,
)

Посчитаем метрики бейзлайна.

In [13]:
display(scores(["baseline"]))

,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate
run,,,,,
baseline,0.1741,0.2422,0.106,0.0391,0.4375


## Квантизация (5 баллов)

Используя код с семинара выполните следующие задания:
1) На примере одного из методов откалибруйте бейзлайн на MASSIVE train и на WikiText, сравните качество на dev. Есть ли влияние калибровочного датасета на метрику? (1 балл)

2) Поэкспериментируйте с разными конфигурациями методов AWQ, GPTQ, SmoothQuant (3 балла)

    Запустите минимум три конфигурации, пробуйте менять разные факторы:

    - битность
    - число калибровочных примеров;
    - group size
    - symmetric / asymmetric quantization;
    - smoothing_strength
    - Дополнительно (+1 балл) — какие слои оставить в исходной точности

3) Объясните результат — какие параметры наиболее значимы? (1 балл)


In [14]:
for name in QUANT:
    run(name)

quant_config = pd.DataFrame(QUANT).T.fillna("")
quant_config.index.name = "run"
display(quant_config.join(scores(["baseline", *QUANT])))

,method,calibration,scheme,keep_edges,alpha,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate
run,,,,,,,,,,
awq_massive,AWQ,massive,W4A16,,,0.1473,0.2188,0.0759,0.0312,0.7188
awq_wikitext,AWQ,wikitext,W4A16,,,0.1558,0.2383,0.0733,0.0391,0.7266
gptq_massive,GPTQ,massive,W4A16,,,0.1633,0.2266,0.1000,0.0273,0.4805
gptq_edges_bf16,GPTQ,massive,W4A16,True,,0.1046,0.1562,0.0529,0.0312,0.3828
smoothquant_massive,SmoothQuant,massive,W8A8,,0.5,0.2083,0.2617,0.1549,0.0391,0.4922


### Выводы

В паре AWQ WikiText дал 15,58% Quality против 14,73% на MASSIVE. В обоих запусках по 128 примеров и 64 474 токена. Сохранение первого и последнего блока в BF16 не помогло GPTQ. Quality снизился с 16,33% до 10,46%.

На этом dev лучший из сжатых вариантов — SmoothQuant W8A8 с Quality 20,83% и размером весов 2,66 ГБ. У бейзлайна 17,41% и 4,06 ГБ. Сравнивались готовые конфигурации, у SmoothQuant меняется и метод, и битность.

## Hard-label дистилляция (5 баллов)

Теперь реализуем hard-label дистилляцию знаний от учителя **Qwen3-32B** в бейзлайн.

### Задания
1) Измерьте качество учителя на dev и сравните с бейзлайном: насколько учитель сильнее? (1 балл)
2) Получите жадные ответы учителя на сэмпле из train. Используйте батч для ускорения при необходимости (1 балл)
3) Реализуйте код для обучения ученика на ответах учителя (1 балл)
4) Проведите эксперименты с обучением, получите модель ученика, отклоняющуюся на dev не более чем на 10% от метрики учителя (2 балла)


### Измерим качество учителя и сравним с бейзлайном (1 балл)

In [15]:
print(TEACHER_MODEL, TEACHER_REVISION)

Qwen/Qwen3-32B 9216db5781bf21249d130ec9da846c4624c16137


In [16]:
run("teacher")

In [17]:
display(scores(["baseline", "teacher"]))

,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate
run,,,,,
baseline,0.1741,0.2422,0.1060,0.0391,0.4375
teacher,0.5805,0.7852,0.3759,0.3125,0.9766


### Получим ответы учителя на трейне (1 балл)

Не забывайте выгружать модели и очищать память.

In [18]:
answers_path = ARTIFACTS / "teacher_answers" / "answers.jsonl"
answers = read_rows(answers_path)
teacher_info = json.loads(answers_path.with_name("summary.json").read_text())
print(f'{teacher_info["valid"]} валидных ответов из {len(answers)}')

1994 валидных ответов из 2048


In [19]:
first = answers[0]
print(first["text"])
print(first["teacher_answer"])

закажи для меня чизбургер в ам бургере
{"intent":"takeaway_order","slots":[["food_type","чизбургер"],["business_name","ам бургер"]]}


### Реализация пайплайна обучения ученика (1 балл)

Полное дообучение Qwen3-8B на A100 80GB проходит на грани по памяти. Для решения можно включать gradient checkpointing, уменьшать batch или брать меньшую модель того же семейства.

In [20]:
sample = TeacherAnswerDataset(answers)[0]
print("токенов ответа со вкладом в loss", sum(x != -100 for x in sample["labels"]))

токенов ответа со вкладом в loss 28


In [21]:
pd.DataFrame(STUDENTS).T

,samples,lr,epochs
sft_512,512.0,0.00002,2.0
sft_2048,2048.0,0.00002,2.0


In [22]:
run("sft_512")
display(scores(["baseline", "teacher", "sft_512"]))

,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate
run,,,,,
baseline,0.1741,0.2422,0.1060,0.0391,0.4375
teacher,0.5805,0.7852,0.3759,0.3125,0.9766
sft_512,0.5645,0.7461,0.3828,0.3125,0.9883


### Добейтесь максимального качества ученика. Цель — не более 10% отклонения от учителя (2 балла)
Пробуйте менять гиперпараметры, число сэмплов, модели ученика (при условии размера модели ученика не более 8B).

In [23]:
for name in STUDENTS:
    run(name)

selection_path = ARTIFACTS / "student_selection.json"
if not selection_path.is_file():
    select_student()
selection = json.loads(selection_path.read_text())
display(scores(["baseline", "teacher", *STUDENTS]))
print("лучший ученик", selection["run"])
print(f'доля качества учителя {selection["quality"] / result("teacher")["quality"]:.2%}')

for name in ["baseline", "teacher", *QUANT, *STUDENTS]:
    predictions = read_rows(ARTIFACTS / "eval" / name / "predictions.jsonl")
    assert [row["id"] for row in predictions] == list(dev_data["id"])
    check = compute_metrics(
        [row["prediction"] for row in predictions], dev_data["gold"], intents, slot_types,
    )
    assert all(check[key] == result(name)[key] for key in METRICS)
print("метрики пересчитаны из ответов на dev")

,quality,intent_accuracy,slot_micro_f1,frame_exact_match,valid_json_rate
run,,,,,
baseline,0.1741,0.2422,0.1060,0.0391,0.4375
teacher,0.5805,0.7852,0.3759,0.3125,0.9766
sft_512,0.5645,0.7461,0.3828,0.3125,0.9883
sft_2048,0.6110,0.8047,0.4172,0.3320,1.0000


лучший ученик sft_2048
доля качества учителя 105.24%


метрики пересчитаны из ответов на dev


### Выводы

Для обеих конфигураций ученик стартует с Qwen3-1.7B. Loss считается на токенах валидных ответов учителя. Первого прогона уже хватило для порога в 90% качества учителя. С увеличением выборки Quality вырос с 56,45% до 61,10%. Учитель на том же dev получил 58,05%. Как student_sft сохранён запуск с 2048 генерациями, в обучение вошли 1994 из них.

### Оценка

Полностью выполненная работа оценивается в 10 баллов. При невыполнении части заданий баллы снимаются в соответствии с критериями:

**Квантизация — 5 баллов**

1. Калибровка бейзлайна на MASSIVE train и на WikiText, влияние калибровочного датасета — 1 балл.
2. Эксперименты с разными конфигурациями методов AWQ, GPTQ, SmoothQuant — 3 балла.
    - По 1 баллу за каждую конфигурацию, которая квантизована и оценена на dev, максимум 3 балла.
    - Бонус +1 балл: часть слоёв оставлена в исходной точности, результат сравнён с аналогичной конфигурацией без исключений слоёв.
3. Объяснение результата — 1 балл.

**Hard-label дистилляция — 5 баллов**

1. Качество учителя на dev и сравнение с бейзлайном — 1 балл.
2. Ответы учителя на трейне — 1 балл.
3. Реализация пайплайна обучения ученика — 1 балл.
4. Обучение ученика — 2 балла.
    - 2 балла: на dev качество ученика не ниже 90% от качества учителя: $Q_{\mathrm{student}} \geq 0.9\,Q_{\mathrm{teacher}}$.
    - 1 балл: цель не достигнута, но ученик лучше бейзлайна на dev и проведено несколько экспериментов с гиперпараметрами, числом примеров или моделью ученика.
    - 0 баллов: ученик не лучше бейзлайна или экспериментов нет.

Баллы за пункт ставятся только при наличии кода, который воспроизводит приведённые метрики, и вывода, опирающегося на эти метрики.